In [26]:
import pandas as pd
import numpy as np 

In [4]:
# Load in core files into dataframes
results = pd.read_csv('../data/raw/results.csv')
races = pd.read_csv('../data/raw/races.csv')
pit_stops = pd.read_csv('../data/raw/pit_stops.csv')
drivers = pd.read_csv('../data/raw/drivers.csv')  
constructors = pd.read_csv('../data/raw/constructors.csv')

## Sanity Check: Core File Loading

All 5 core CSV files loaded successfully from `data/raw/`.

| File | Rows | Columns | Notes |
|------|------|---------|-------|
| results | 26,759 | 18 | Row count matches data audit. All columns appear non-null because `\N` is stored as text. `position`, `number`, `time`, `milliseconds`, `fastestLap`, `rank`, `fastestLapTime`, `fastestLapSpeed` are read as string type and will need conversion after `\N` replacement. |
| races | 1,125 | 18 | `\N` visible in `head()` for practice session, qualifying, and sprint columns. `year` is int64, ready for filtering. |
| pit_stops | 11,371 | 7 | `duration` loaded as string, needs float conversion. Data starts 2011 not 2010. |
| drivers | 861 | 9 | `number` and `code` are strings, will contain `\N` for pre-2014 drivers. |
| constructors | 212 | 5 | Cleanest file. No issues detected. |

**Key finding:** Every column across all 5 files shows 0 nulls. This is misleading. The dataset uses `\N` as a text placeholder instead of true null values, so pandas reads them as valid strings. Once `\N` is replaced with proper nulls, null counts will change significantly, especially `position` in results.csv which should be approximately 40% null due to DNFs.

**Next step:** Filter to 2010 to 2024 using `races.year`, then replace `\N` text values with proper nulls.

In [21]:
races.columns

Index(['raceId', 'year', 'round', 'circuitId', 'name', 'date', 'time', 'url',
       'fp1_date', 'fp1_time', 'fp2_date', 'fp2_time', 'fp3_date', 'fp3_time',
       'quali_date', 'quali_time', 'sprint_date', 'sprint_time'],
      dtype='object')

In [23]:
# Filter races to the 2010 to 2024 analysis window
races = races[(races['year'] >= 2010) & (races['year'] <= 2024)].copy()

# Build the list of valid race IDs from the filtered races
valid_race_ids = races['raceId']

# Keep only rows in the other files that belong to those races
results = results[results['raceId'].isin(valid_race_ids)].copy()
pit_stops = pit_stops[pit_stops['raceId'].isin(valid_race_ids)].copy()

# Trim the lookup tables to only drivers and constructors who appear in the window
drivers = drivers[drivers['driverId'].isin(results['driverId'])].copy()
constructors = constructors[constructors['constructorId'].isin(results['constructorId'])].copy()

# Sanity check row counts after filtering
print(f"races: {races.shape}")
print(f"results: {results.shape}")
print(f"pit_stops: {pit_stops.shape}")
print(f"drivers: {drivers.shape}")
print(f"constructors: {constructors.shape}")

races: (305, 18)
results: (6436, 18)
pit_stops: (11371, 7)
drivers: (80, 9)
constructors: (23, 5)


## Filtering to the 2010 to 2024 Analysis Window

Filtered all 5 core files to only include races from 2010 to 2024. Since only `races.csv` has a `year` column, the filter flows through `raceId`:

1. Filter `races` on `year` between 2010 and 2024
2. Extract valid `raceId` values from the filtered `races`
3. Filter `results` and `pit_stops` using `.isin()` on `raceId`
4. Filter `drivers` and `constructors` using `.isin()` on the driver and constructor IDs found in the filtered `results`

`.copy()` is applied after each filter to create independent DataFrames, preventing `SettingWithCopyWarning` when modifying columns in later steps.

### Row Counts After Filtering

| File | Rows | Columns |
|------|------|---------|
| races | 305 | 18 |
| results | 6,436 | 18 |
| pit_stops | 11,371 | 7 |
| drivers | 80 | 9 |
| constructors | 23 | 5 |

Row counts are consistent with expectations: 305 races across 15 seasons (about 20 per year), 6,436 results at roughly 20 drivers per race, pit_stops unchanged since the data was already 2011 onward, and drivers and constructors reduced to only those active in this window.

In [ ]:
# Replace \N with proper nulls across all 5 DataFrames, write a loop

# Create a dictionary of the dataframes to iterate over
dataframes = {
    'races': races,
    'results': results,
    'pit_stops': pit_stops,
    'drivers': drivers,
    'constructors': constructors
}

# Loop through the dictionary and replace \N with pd.NA in each dataframe
for name, df in dataframes.items():
    dataframes[name] = df.replace('\\N', pd.NA)

races = dataframes['races']
results = dataframes['results']
pit_stops = dataframes['pit_stops']
drivers = dataframes['drivers']
constructors = dataframes['constructors']

In [30]:
# Loop through the dictionary and print out null summary for each dataframe
for name, df in dataframes.items():
    print(f"\n{'='*50}")
    print(f"{name} null summary:")
    print(f"{'='*50}")
    print(df.isnull().sum())


races null summary:
raceId           0
year             0
round            0
circuitId        0
name             0
date             0
time             0
url              0
fp1_date       215
fp1_time       237
fp2_date       215
fp2_time       237
fp3_date       233
fp3_time       252
quali_date     215
quali_time     237
sprint_date    287
sprint_time    290
dtype: int64

results null summary:
resultId              0
raceId                0
driverId              0
constructorId         0
number                0
grid                  0
position           1039
positionText          0
positionOrder         0
points                0
laps                  0
time               3126
milliseconds       3126
fastestLap          285
rank                 27
fastestLapTime      285
fastestLapSpeed     285
statusId              0
dtype: int64

pit_stops null summary:
raceId          0
driverId        0
stop            0
lap             0
time            0
duration        0
milliseconds    0
dtype

## Null Audit After Replacing `\N`

After replacing text `\N` values with proper nulls, the null landscape breaks into three categories: structural nulls that carry meaning, nulls in columns that are already scoped for removal, and clean files with no nulls.

### races
All 10 columns with nulls (fp1_date, fp1_time, fp2_date, fp2_time, fp3_date, fp3_time, quali_date, quali_time, sprint_date, sprint_time) are already scoped as skip columns. Nulls will be resolved when these columns are dropped in the next step.

### results
| Column | Nulls | Decision | Reason |
|--------|-------|----------|--------|
| position | 1,039 | Leave as null | Structural. Represents DNFs (did not finish). Will be excluded from `position_change` calculation and documented as a modeling limitation. |
| time, milliseconds | 3,126 each | Drop with column | Already scoped as skip. Race time is only clean for the winner, everyone else has a relative gap. |
| fastestLap, fastestLapTime, fastestLapSpeed | 285 each | Drop with column | Already scoped as skip. |
| rank | 27 | Drop with column | Already scoped as skip. |

### pit_stops
No nulls. Clean file.

### drivers
`number` has 21 nulls. This reflects a real-world F1 rule change: permanent driver numbers were only introduced in 2014, so drivers who raced only before 2014 have no permanent number. Decision: leave as null. This is a structural null tied to a documented rule change, not a data quality issue.

### constructors
No nulls. Clean file.

### Summary
No rows will be dropped for null handling. No values will be imputed. Structural nulls are preserved because they carry information (DNFs, pre-2014 driver numbers). All other nulls are in columns already scoped for removal.

In [ ]:
# Create a dictionary of the dataframes to use only keep columns that are needed for the analysis
keep_columns = {
    'results': ['raceId', 'driverId', 'constructorId', 'grid', 'position', 'positionText', 'points', 'statusId'],
    'races': ['raceId', 'year', 'circuitId', 'name'], 
    'pit_stops': ['raceId', 'driverId', 'stop', 'lap', 'duration'],
    'drivers': ['driverId', 'driverRef', 'number', 'code', 'forename', 'surname', 'nationality'],
    'constructors': ['constructorId', 'constructorRef', 'name', 'nationality']
}

In [ ]:
# Rename columns for clarity and consistency across dataframes
rename_map = {
    'races': {'name': 'race_name'},
    'constructors': {'name': 'constructor_name'}
}

In [35]:
# For each entry in keep_columns, grab the name and the column list. Go to the matching DataFrame in dataframes, filter it to those columns, and save the copy back
for name, cols in keep_columns.items():
    dataframes[name] = dataframes[name][cols].copy()

In [36]:
races = dataframes['races']
results = dataframes['results']
pit_stops = dataframes['pit_stops']
drivers = dataframes['drivers']
constructors = dataframes['constructors']

In [37]:
print(f"races: {races.shape}")
print(f"results: {results.shape}")
print(f"pit_stops: {pit_stops.shape}")
print(f"drivers: {drivers.shape}")
print(f"constructors: {constructors.shape}")

races: (305, 4)
results: (6436, 8)
pit_stops: (11371, 5)
drivers: (80, 7)
constructors: (23, 4)


In [ ]:
# Create a for loop for rename columns
for names, mapping in rename_map.items():
    dataframes[names] = dataframes[names].rename(columns=mapping)

In [39]:
races = dataframes['races']
constructors = dataframes['constructors']